In [27]:
import os
from dotenv import load_dotenv
load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = google_api_key

In [ ]:

from langgraph.prebuilt import create_react_agent
from langchain.chat_models import init_chat_model

In [28]:
llm = init_chat_model( model="gemini-2.5-flash",model_provider="google-genai")

In [54]:
import sqlite3

def database_tool(query):
    """Does CRUD operations based on the SQL query generated by the agent.
    users table schema: (email TEXT PRIMARY KEY, name TEXT, age INTEGER)
    """
    conn = sqlite3.connect('db/first.db')
    c = conn.cursor()
    try:
        c.execute(query)
        conn.commit()
        result = "User created successfully."
    except sqlite3.IntegrityError as e:
        result = f"Error: {e}"
    conn.close()
    return result

In [55]:
res=database_tool ("INSERT INTO users (email, name, age) VALUES (\'pratap@gmail.com\', \'Piyush Pratap\', 25)")
res

'Error: UNIQUE constraint failed: users.email'

In [56]:
custom_prompt= """You are a database assistant. You perform CRUD operations on users table with schema (email TEXT PRIMARY KEY, name TEXT, age INTEGER) based on the user query. 
- Extract the relevant information from the user queries to form the SQL queries. 
- If the user query does not have complete information, ask for the missing information then perform the operation. 
- Always use parameterized queries to prevent SQL injection."""

In [ ]:
agent = create_react_agent(llm, [database_tool], prompt=custom_prompt)

In [58]:
user_query = "My name is Ram Kishan and my email is ramk@gmail.com and I am 21 years old. Please add me to the database."

In [59]:
res = agent.invoke({"messages": [("user", user_query)]})
print(res)


{'messages': [HumanMessage(content='My name is Ram Kishan and my email is ramk@gmail.com and I am 21 years old. Please add me to the database.', additional_kwargs={}, response_metadata={}, id='dab7bd4c-db9b-4feb-a3ca-7abbaa37eca4'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'database_tool', 'arguments': '{"query": "INSERT INTO users (email, name, age) VALUES (\'ramk@gmail.com\', \'Ram Kishan\', 21)"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--55298664-4b87-4582-b78c-c9896dffe763-0', tool_calls=[{'name': 'database_tool', 'args': {'query': "INSERT INTO users (email, name, age) VALUES ('ramk@gmail.com', 'Ram Kishan', 21)"}, 'id': '431b0e2b-f050-40a4-bdab-544bcb716eff', 'type': 'tool_call'}], usage_metadata={'input_tokens': 179, 'output_tokens': 236, 'total_tokens': 415, 'input_token_details': {'cache_read': 0}, 'output_token_details'

In [61]:
res["messages"][-1].content

'User created successfully.'

In [63]:
update_query = "My name is Piyush Pratap and my email is pratap@gmail.com. I want to update my age to 24."

In [64]:
res1 = agent.invoke({"messages": [("user", update_query)]})
print(res1["messages"][-1].content)

I have updated your age to 24.
